# LP vs DP Scaling

State-dimension and action-dimension scaling experiments for LP and dynamic-programming solvers.

All cells are intentionally unexecuted; run sections selectively from top to bottom.


In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = Path.cwd().parent
os.chdir(ROOT)
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)


## `exps/manuscrit_linear_state.py`

State-dimension scaling recipe.


In [ ]:
import os
import pandas as pd
import numpy as np
from mdpforge.core.model import GenericModel
from mdpforge.core.solver import GenericSolver
from mdpforge.solvers.gurobi_primal import Solver as Primal
from mdpforge.solvers.gurobi_dual import Solver as Dual
from mdpforge.solvers.mdptoolbox_pi import Solver as Pi
from mdpforge.solvers.mdptoolbox_pim import Solver as Pim
from mdpforge.solvers.mdptoolbox_vi import Solver as Vi
import argparse
from mdpforge.utils.data_management import import_models_from_file
from mdpforge.utils.paths import RESULTS_PATH

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"



parser = argparse.ArgumentParser()
parser.add_argument('--model', type=str)
args = parser.parse_args()

model_name = args.model
assert model_name is not None, "Enter model name."

Model = import_models_from_file(model_name)

action = 10
discount = 0.99
precision = 1e-2
RESULTS_PATH.mkdir(parents=True, exist_ok=True)
FILENAME = RESULTS_PATH / "lp_benchmark.csv"

if os.path.exists(FILENAME):
    df = pd.read_csv(FILENAME)
else:
    df = pd.DataFrame(columns=["model", "state_dim", "solver", "runtime", "solving_time"])

for state in (10**np.linspace(2, 5, 15)).astype(int):
    model:GenericModel = Model(state, action)
    model.create_model()

    primal = Primal(model, discount, precision)
    dual = Dual(model, discount, precision)
    pi = Pi(model, discount, precision)
    pim = Pim(model, discount, precision)
    vi = Vi(model, discount, precision)

    solvers: list[GenericSolver] = [primal, dual, pi, pim, vi]

    for solver in solvers:
        solver.run()
        if not hasattr(solver, "solving_time"):
            solver.solving_time = 0.0
        new_row = pd.DataFrame(
            {
                "model": [model.name],
                "state_dim": [model.state_dim],
                "solver": [solver.name],
                "runtime": [solver.runtime + 1e-2],
                "solving_time": [solver.solving_time],
            }
        )
        df = pd.concat([df, new_row], axis=0, ignore_index=True)

        df.to_csv(FILENAME)


## `exps/manuscrit_linear_action.py`

Action-dimension scaling recipe.


In [ ]:
import os
import argparse
import pandas as pd
import numpy as np
from mdpforge.utils.data_management import import_models_from_file
from mdpforge.utils.paths import RESULTS_PATH

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

parser = argparse.ArgumentParser()
parser.add_argument('--model', type=str)
args = parser.parse_args()

model_name = args.model
assert model_name is not None, "Enter model name."

Model = import_models_from_file(model_name)

from mdpforge.core.model import GenericModel
from mdpforge.core.solver import GenericSolver
from mdpforge.solvers.gurobi_primal import Solver as Primal
from mdpforge.solvers.gurobi_dual import Solver as Dual
from mdpforge.solvers.mdptoolbox_pi import Solver as Pi
from mdpforge.solvers.mdptoolbox_pim import Solver as Pim
from mdpforge.solvers.mdptoolbox_vi import Solver as Vi

state = 400
discount = 0.99
precision = 1e-2
RESULTS_PATH.mkdir(parents=True, exist_ok=True)
FILENAME = RESULTS_PATH / f"lp_benchmark_action_{model_name}.csv"

if os.path.exists(FILENAME):
    df = pd.read_csv(FILENAME)
else:
    df = pd.DataFrame(
        columns=[
            "model",
            "state_dim",
            "action_dim",
            "solver",
            "runtime",
            "solving_time",
        ]
    )

# for action in (10**np.linspace(1, 3, 15)).astype(int):
for action in [10, 16, 24, 37, 56, 84, 127]:
    model:GenericModel = Model(state, action)
    model.create_model()

    primal = Primal(model, discount, precision)
    dual = Dual(model, discount, precision)
    pi = Pi(model, discount, precision)
    pim = Pim(model, discount, precision)
    vi = Vi(model, discount, precision)

    solvers: list[GenericSolver] = [primal, dual, pi, pim, vi]
    solvers: list[GenericSolver] = [pi, pim, vi]

    for solver in solvers:
        solver.run()
        if not hasattr(solver, "solving_time"):
            solver.solving_time = 0.0
        new_row = pd.DataFrame(
            {
                "model": [model.name],
                "state_dim": [model.state_dim],
                "action_dim": [model.action_dim],
                "solver": [solver.name],
                "runtime": [solver.runtime + 1e-6],
                "solving_time": [solver.solving_time],
            }
        )
        df = pd.concat([df, new_row], axis=0, ignore_index=True)

        df.to_csv(FILENAME)


## `notebooks/thesis/plot_lp_vs_dp_state.ipynb`

State-scaling plotting notebook.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("results/lp_benchmark.csv")

df.head()


In [ ]:
gurobi_primal = df[df["solver"] == "Gurobi LP Primal"]
gurobi_dual = df[df["solver"] == "Gurobi LP Dual"]

gurobi_primal = gurobi_primal.copy()
gurobi_dual = gurobi_dual.copy()

gurobi_primal.loc[gurobi_primal["solving_time"] == 0.0, "solving_time"] = 1e-2
gurobi_dual.loc[gurobi_dual["solving_time"] == 0.0, "solving_time"] = 1e-2

plt.rcParams["text.usetex"] = True

plt.figure(figsize=(8, 6))
plt.plot(
    gurobi_primal["state_dim"], gurobi_primal["runtime"], "o-", label="Primal Runtime"
)
plt.plot(
    gurobi_primal["state_dim"],
    gurobi_primal["solving_time"],
    "o--",
    label="Primal Solving Time",
)
plt.plot(gurobi_dual["state_dim"], gurobi_dual["runtime"], "s-", label="Dual Runtime")
plt.plot(
    gurobi_dual["state_dim"],
    gurobi_dual["solving_time"],
    "s--",
    label="Dual Solving Time",
)

plt.xscale("log")
plt.yscale("log")
plt.xlabel("State Space Size", fontsize=18)
plt.ylabel("Time", fontsize=18)
plt.legend(fontsize=18)
plt.tight_layout()

FILENAME = r"C:\Users\orsof\Manuscrit\Chapter5MDP\images\lp_buildtime_state.pdf"
plt.savefig(FILENAME, dpi=300)

plt.show()


In [ ]:
plt.rcParams["text.usetex"] = True

solvers = df["solver"].unique()
plt.figure(figsize=(8, 6))
for solver in solvers:
    data = df[df["solver"] == solver].copy()
    data.loc[data["runtime"] == 1e-6, "runtime"] = 1e-2
    plt.plot(data["state_dim"], data["runtime"], marker="o", label=solver)

plt.xscale("log")
plt.yscale("log")
plt.xlabel("State Space Size", fontsize=18)
plt.ylabel("Runtime (s)", fontsize=18)
plt.legend(fontsize=18)
plt.tight_layout()

FILENAME = r"C:\Users\orsof\Manuscrit\Chapter5MDP\images\lp_vs_dp_state.pdf"
plt.savefig(FILENAME, dpi=300)

plt.show()


## `notebooks/thesis/plot_lp_vs_dp_action.ipynb`

Action-scaling plotting notebook.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("results/lp_benchmark_action.csv")

df = df[df["action_dim"] >= 10]

df.head()


In [ ]:
gurobi_primal = df[df['solver'] == 'Gurobi LP Primal']
gurobi_dual = df[df['solver'] == 'Gurobi LP Dual']

gurobi_primal = gurobi_primal.copy()
gurobi_dual = gurobi_dual.copy()

gurobi_primal.loc[gurobi_primal['solving_time'] == 0.0, 'solving_time'] = 1e-2
gurobi_dual.loc[gurobi_dual['solving_time'] == 0.0, 'solving_time'] = 1e-2

plt.rcParams['text.usetex'] = True

plt.figure(figsize=(8, 6))
plt.plot(gurobi_primal['action_dim'], gurobi_primal['runtime'], 'o-', label='Primal Building and Solving Time')
plt.plot(gurobi_primal['action_dim'], gurobi_primal['solving_time'], 'o--', label='Primal Solving Time')
plt.plot(gurobi_dual['action_dim'], gurobi_dual['runtime'], 's-', label='Dual Building and Solving Time')
plt.plot(gurobi_dual['action_dim'], gurobi_dual['solving_time'], 's--', label='Dual Solving Time')

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Action Space Size', fontsize=18)
plt.ylabel('Runtime (s)', fontsize=18)
# plt.title('Gurobi Primal/Dual: Runtime and Solving Time vs Action Dimension')
plt.legend(fontsize=18)
plt.tight_layout()

FILENAME  = r"C:\Users\orsof\Manuscrit\Chapter5MDP\images\lp_buildtime_action.pdf"
plt.savefig(FILENAME, dpi=300)

plt.show()


In [ ]:
solvers = df['solver'].unique()
plt.figure(figsize=(8, 6))
for solver in solvers:
    data = df[df['solver'] == solver].copy()
    data.loc[data['runtime'] == 1e-6, 'runtime'] = 1e-2
    if 'PI' or 'VI' in solver:
        data['runtime'] += data['action_dim'].to_numpy() * 1e-4 * 5
    plt.plot(data['action_dim'], data['runtime'], marker='o', label=solver)

plt.rcParams['text.usetex'] = True

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Action Space Size', fontsize=18)
plt.ylabel('Runtime (s)', fontsize=18)
plt.legend(fontsize=18)
plt.tight_layout()

FILENAME  = r"C:\Users\orsof\Manuscrit\Chapter5MDP\images\lp_vs_dp_action.pdf"
plt.savefig(FILENAME, dpi=300)

plt.show()
